# Import and Shared functions

In [ ]:
import os, sys
import math
import re
import pandas as pd
import numpy as np
import ephem
from scipy import signal
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
from plotly.subplots import make_subplots
from dateutil import parser as dateutil_parser
sys.path.insert(0, os.path.abspath(os.path.join('.', '..', 'driver')))
import importlib
import kinematics
import control
import analyse_helpers
importlib.reload(kinematics)
importlib.reload(control)
importlib.reload(analyse_helpers)
from kinematics import calc_parallactic_angle, azaltroll_to_theta, apply_mechanical_corrections, azaltroll_to_q, MountModelParams
from control import theta_to_jacobian, PecMode
from quaternion import Q as Quaternion
from analyse_helpers import resolve_log_files, load_pec, load_kf_pid, load_phd2_guidelog, window_df, load_sync_guiding_residuals
from analyse_helpers import frame_shift_components, ramp_vs_next_step, correction_breakdown, lombscargle_periodogram
from catalog_logs import catalog_session
import pe_analysis; importlib.reload(pe_analysis)
from pe_analysis import replay_pec_rate, rate_scores, true_rate, PecModel
from extract_segments import stitch_accum
from IPython.display import display

ARCSEC = 3600  # deg -> arcsec, same convention as analyse_kf_pid.ipynb

def r2_score(y, yhat):
    ss_res = np.sum((y - yhat)**2)
    ss_tot = np.sum((y - np.mean(y))**2)
    return 1 - ss_res / ss_tot if ss_tot != 0 else np.nan

def r2_quality(r2):
    return (
        "excellent" if r2>0.95 else 
        "good" if r2>0.8 else 
        "moderate" if r2>0.5 else 
        "weak" if r2>0.25 else 
        "poor")

def pec_quality_db(snr_db):
    return (
        "excellent" if snr_db > 30 else
        "good"      if snr_db > 20 else
        "moderate"  if snr_db > 10 else
        "weak"      if snr_db > 5  else
        "poor"
    )


# Load data

In [ ]:
# ── Choose the session to analyse ─────────────────────────────────────────────────────────────────────
# Every captured log is registered once in utility/sessions.toml: its files, PHD2 guide log, default time window and notes.
# New logs: copy them into logs/archive, run `uv run python sessions.py --add-new` in utility/, then edit the notes.
# What's available:  list_sessions()  /  list_sessions(records='pec')  /  list_sessions(contains='eagle')
from sessions import get_session, list_sessions
SESSION = 'jdm_Beta7.1_09_29_pulseguide'
S = get_session(SESSION)
log_filenames, LOG_DIR = S.log_filenames, S.log_dir
PHD2_GUIDELOG = S.phd2              # None skips the "Guider Outcome" section; everything else only needs the driver log

# ── Optional time window (local time, as written in the log) -- None = whole log ─────────────────────
# Defaults to the session's t_from / t_to in sessions.toml; override here to cut a night down to one PEC run.
T_FROM, T_TO = S.t_from, S.t_to

# ── Optional: analyse one automatically-detected tracking segment (listed when the log loads) ────────
# None = use T_FROM / T_TO above; an index (e.g. 3) = that segment's start/end (overrides T_FROM / T_TO).
SEGMENT = None

print(f"{SESSION}: {S.notes}\n  files {S.log_filenames}  PHD2 {'yes' if PHD2_GUIDELOG else 'no'}  window {T_FROM}..{T_TO}")


In [ ]:
# ── Read and Parse PECLOG data from log file(s) ───────────────────────────────────────────────────────────
resolved_files = resolve_log_files(log_filenames, log_dir=LOG_DIR)
print(f"Loading {len(resolved_files)} file(s) from LOG_DIR={LOG_DIR!r}:")
for p in resolved_files:
    print(f"  {p}  ({os.path.getsize(p)/1e6:.1f} MB)")
print()

df, pec_config = load_pec(log_filenames, log_dir=LOG_DIR)

# tracking segments of this log (split at gotos, slews, jogs, rotates, parks, tracking changes), as in the catalog
SEGMENTS = pd.DataFrame(catalog_session(resolved_files))
if len(SEGMENTS):
    print("Tracking segments in this log (set SEGMENT = index in the log-selection cell to analyse one):")
    seg_view = SEGMENTS[SEGMENTS['duration_min'] >= 1][['start', 'end', 'duration_min', 'start_reason', 'end_reason', 'pec',
                                                        'n_sync_guide', 'pulse_guiding', 'alt_mean', 'roll_mean']].copy()
    seg_view['start'] = seg_view['start'].dt.strftime('%m-%d %H:%M'); seg_view['end'] = seg_view['end'].dt.strftime('%m-%d %H:%M')
    display(seg_view.round(1))                      # index = SEGMENT number (segments under 1 min hidden)
if SEGMENT is not None:
    T_FROM, T_TO = SEGMENTS.loc[SEGMENT, 'start'], SEGMENTS.loc[SEGMENT, 'end']
    print(f"Using segment {SEGMENT}: {T_FROM} .. {T_TO}")
df = window_df(df, T_FROM, T_TO)
if T_FROM or T_TO:
    print(f"Window {T_FROM} .. {T_TO}: {len(df)} PECLOG rows")
if df.empty:
    raise ValueError(f"No PECLOG rows inside the window T_FROM={T_FROM!r} T_TO={T_TO!r}")

# Fallback if this log predates PECCONFIG (older sessions) — infer what we can
if pec_config is None:
    ra_model_cols = [c for c in df.columns if re.fullmatch(r'ra_model_\d+', c)]
    n_model_terms = len(ra_model_cols)
    if n_model_terms == 0:
        print("WARNING: no PECCONFIG line and no ra_model/dec_model columns found — "
              "cannot determine mode reliably. Defaulting to RLS/H=0, verify manually.")
        pec_config = dict(mode='rls', H=0, T=34*60, tau=21*60, min_dt=0.05)
    else:
        pec_config = dict(mode='rls', H=max(n_model_terms - 1, 0), T=34*60, tau=21*60, min_dt=0.05)
    print("No PECCONFIG line found — inferred:", pec_config)
else:
    print("PEC config from log:", pec_config)

print(f"Duration: {df['t_sec'].iloc[-1]/3600:.2f} hours")
print(f"N samples: {len(df)}")
print(f"Az range: {df['az'].min():.3f} to {df['az'].max():.3f}")
print()
df.columns


# Optional Save to CSV

In [ ]:
# df.to_csv('data.csv', index=False)

# Mount Orientation (Az, Alt, Roll) vs Time

These three plots show how Azimuth, Altitude and Roll Angle are changing over the session. 
Ideally they should trace smooth curves that are following the sidereal motion. If there are large steps then consider
dividing the log file into separate sessions as the mount has likely moved to a different target.

In [ ]:
df.columns

In [ ]:
# ── plot Az, Alt, Roll ───────────────────────────────────────────────────────────
xdata = df['t_sec'] / 60
xdata = df['timestamp'] 

def wrap_to_median(y, ref=None):
    """Re-express a circular (0-360) quantity as a signed deviation from `ref` (default: the
    data's own median), wrapped into (-180, 180] around it, then ref added back -- e.g. if
    the bulk of the samples sit near 0 but a few sit at 359, those become -1 instead of the
    0's staying put and the axis stretching out to fit 359. Using the plain numeric median
    (not a circular mean via atan2) is deliberate: atan2 always returns its result on the
    (-180, 180] branch, so a cluster that's genuinely near (say) 270 or 340 would come back
    as -90 or -20 and get added back at the *wrong* branch, pulling the whole recentered
    trace down near there instead of leaving it near 270/340. The median has no such branch
    to pick wrong -- it's just whichever raw sample sits in the middle once sorted, so as long
    as a wrap crossing is a minority of the samples (the normal case), it lands inside the
    real majority cluster on its own, wherever on the circle that happens to be. This also
    makes the trace continuous through a normal wrap -- the discontinuity was only ever in
    the 0-360 coordinate, not the mount's actual motion."""
    if ref is None:
        ref = y.median()
    return (y - ref + 180) % 360 - 180 + ref

def break_wraps(y, threshold=180):
    """NaN out a sample where it jumps by more than `threshold` degrees from the previous one
    so a line plot doesn't draw a spurious near-vertical segment straight across it. Run
    *after* wrap_to_median() -- that already makes an ordinary 0/360 wrap continuous, so
    anything still jumping this much here is a real, separate discontinuity (e.g. a slew to a
    new target -- see the markdown note above) worth flagging as a gap rather than a smooth
    line."""
    y = y.copy()
    y[y.diff().abs() > threshold] = np.nan
    return y

az_ref = df['az'].median()
print(f"Azimuth wrapped around its median ({az_ref:.2f}°)")

ydata = [
    #row, dataset,                                          name,                color
    (1,   break_wraps(wrap_to_median(df['az'], az_ref)),    'Azimuth (°, wrapped)', 'royalblue'),
    (2,   df['alt'],                                        'Altitude (°)',        'orange'),
    (3,   df['roll'],                                       'Roll (°)',            'mediumseagreen'),
]
rows = len(set([row for row, y,name,color in ydata]))
fig = make_subplots(rows=rows, cols=1, shared_xaxes=True,
    subplot_titles=[name for row, y,name,color in ydata],
    vertical_spacing=0.12)

for (row, y, name, color) in ydata:
    fig.add_trace(go.Scatter(x=xdata, y=y,
        mode='lines+markers', line=dict(color=color, width=0.8),
        name=name), row=row, col=1)
    fig.update_yaxes(title_text=name, row=row, col=1)

fig.update_xaxes(title_text='Time (minutes)', row=3, col=1)
fig.update_layout(height=800, width=1200, template="plotly_dark",
        title='Range of Mount Orientation over session')
fig.show()

# Reading the outcome sections

Each section answers a different question about the same session:

| Section | Question | Source | Caveat |
|---|---|---|---|
| **PEC Outcome Overview** | Are the guide corrections getting smaller over the session? | PECLOG `resid` | Confounded by seeing, pose and guider settings over the night |
| **PEC vs Guider** | Who did the correcting: guider, sync guiding or PEC, and did PEC push with or against the drift? | PECLOG running totals | Bookkeeping of what happened, not what could have happened |
| **Guider Outcome** | Did guiding RMS and guider effort change with PEC on vs off? | PHD2 guide log | The sky truth, but noisy (±15-20% run to run) and needs PEC-off time on the same target |
| **PEC rate benchmark** | How well would each PEC model have predicted this session's drift? | PECLOG `total_accum`, replayed | Counterfactual: scores every model on the same drift, whatever PEC actually ran |
| **Base vs Corrected Frame Shift** | What did the corrections do to each motor, and did the syncs undo PEC? | KFLOG | Diagnostic in motor space; needs `log_position` |

The benchmark uses the same assessment as `analyse_pec_catalog.ipynb` (there across all archived sessions): the drift
left for the guider per correction interval, `y(t+dt) - y(t) - rate(t)*dt`, relative to no PEC.

# PEC Outcome Overview

The real test of whether PEC is working: is `resid` (this sync's raw guide error, in arcmin --
`resid_1`=RA, `resid_2`=Dec after the standard list-flattening) trending toward zero and getting
less noisy over the session? This is the ground-truth guiding outcome PEC is trying to improve,
distinct from `total_accum` (which accumulates continuously by design and never shrinks -- see the
section below) and from `rmse`/`r2` (which describe the *model's own fit quality*, not the
resulting guiding error). Per control.py's `_pec_log()`: "should shrink as PEC improves."

Note `n` (and so `resid`) resets whenever the PEC model resets (goto/rotate/jog/stop-tracking/config
change -- see the reset-aware analysis further below), so a session spanning multiple resets will
show the residual jump back up at each reset before resuming its downward trend within that segment.

See also the Base vs Corrected Frame Shift diagnostic below, which cross-checks this from KFLOG telemetry when available.

In [ ]:
# ── PEC Outcome Overview: are guide residuals shrinking over the session? ──────────────────
# Each axis uses its own rows: a sync-guide row carries both RA and Dec, a pulse-guide row only one.
def axis_resid(col):
    r = df.dropna(subset=[col]).sort_values('t_sec').reset_index(drop=True)
    r['arcsec'] = r[col] * 60
    roll_n = max(5, len(r) // 20)   # ~20 points of smoothing across the session, floor of 5 cycles
    r['rms_roll'] = r['arcsec'].rolling(roll_n, min_periods=1).apply(lambda x: np.sqrt(np.mean(x**2)))
    return r, roll_n

ra_resid,  RA_ROLL_N  = axis_resid('resid_1')
dec_resid, DEC_ROLL_N = axis_resid('resid_2')

def trend_summary(t, y, label):
    """OLS slope of y (arcsec) vs t (sec) -- negative means shrinking/improving -- plus a
    first-fifth vs last-fifth RMS comparison, which is more robust to a single noisy cycle
    than the endpoints alone."""
    if len(t) < 4:
        print(f"{label}: not enough guide corrections ({len(t)}) for a trend")
        return
    slope, intercept = np.polyfit(t, y, 1)
    yhat = np.polyval([slope, intercept], t)
    r2 = r2_score(y, yhat)
    n = len(y)
    k = max(1, n // 5)
    rms_first = np.sqrt(np.mean(y[:k]**2))
    rms_last  = np.sqrt(np.mean(y[-k:]**2))
    pct_change = 100 * (rms_last - rms_first) / rms_first if rms_first != 0 else np.nan
    direction = "decreasing -- PEC improving" if slope < 0 else "increasing -- PEC not converging / worsening"
    print(f"=== {label} guide residual outcome (N={n} corrections) ===")
    print(f"Linear trend:         {slope*3600:+8.3f} arcsec/hr   R²={r2:.3f} ({r2_quality(r2)} fit) -- {direction}")
    print(f"First {k} RMS:          {rms_first:8.3f} arcsec")
    print(f"Last  {k} RMS:          {rms_last:8.3f} arcsec")
    print(f"Change:                {pct_change:+7.1f}%  ({'improvement' if pct_change < 0 else 'regression'})")
    print()

trend_summary(ra_resid.t_sec.values,  ra_resid.arcsec.values,  "RA")
trend_summary(dec_resid.t_sec.values, dec_resid.arcsec.values, "Dec")

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
    subplot_titles=["RA guide residual (arcsec) -- should trend toward 0 as PEC learns",
                     "Dec guide residual (arcsec) -- should trend toward 0 as PEC learns"],
    vertical_spacing=0.1)

fig.add_trace(go.Scatter(x=ra_resid.t_sec/60, y=ra_resid.arcsec, mode='markers', name='RA resid (raw)',
    marker=dict(size=4, color='royalblue', opacity=0.4)), row=1, col=1)
fig.add_trace(go.Scatter(x=ra_resid.t_sec/60, y=ra_resid.rms_roll, mode='lines', name=f'RA rolling RMS (n={RA_ROLL_N})',
    line=dict(color='orange', width=2)), row=1, col=1)

fig.add_trace(go.Scatter(x=dec_resid.t_sec/60, y=dec_resid.arcsec, mode='markers', name='Dec resid (raw)',
    marker=dict(size=4, color='mediumseagreen', opacity=0.4)), row=2, col=1)
fig.add_trace(go.Scatter(x=dec_resid.t_sec/60, y=dec_resid.rms_roll, mode='lines', name=f'Dec rolling RMS (n={DEC_ROLL_N})',
    line=dict(color='orange', width=2)), row=2, col=1)

fig.add_hline(y=0, line=dict(color='gray', width=1, dash='dot'), row=1, col=1)
fig.add_hline(y=0, line=dict(color='gray', width=1, dash='dot'), row=2, col=1)

fig.update_xaxes(title_text='Time (minutes)', row=2, col=1)
fig.update_yaxes(title_text='arcsec', row=1, col=1)
fig.update_yaxes(title_text='arcsec', row=2, col=1)
fig.update_layout(height=700, width=1200, template='plotly_dark',
    title='PEC Outcome: Sync-Guide Residual Trend (raw resid + rolling RMS)',
    hovermode='x unified', legend=dict(groupclick='toggleitem'))
fig.show()

# PEC vs Guider: who corrected the drift?

Three views of the same bookkeeping, per axis, straight from PECLOG (what the driver actually did):

1. **Running totals (arcsec).** The **mount drift** (`total_accum`, joined across PEC model resets) with its quadratic
   trend dashed, against the running totals of each correction: **pulse-guide corrections** (PHD2), **sync-guide
   corrections** (plate-solve syncs) and **PEC applied**. By construction mount drift = pulse + sync + PEC.
2. **Rates (arcsec/min).** The same as rates (each a ±5 min slope): actual drift rate = pulse + sync + PEC rates.
   When PEC is working its rate carries the drift rate and the guider's rates sit near zero. When the guider's rate
   mirrors PEC's, the guider is undoing PEC.
3. **PEC rate terms (arcsec/min).** The rate PEC's model gave, split into the **steady-drift term** and the **signed
   harmonic contribution** at that moment (RLS only; EMA has just the one term), with the **PEC applied rate** (zero
   while PEC is inhibited, shaded) and the **actual drift rate** for reference. The gap between model rate and actual
   drift rate is PEC's error.

Dotted vertical lines are PEC model resets (goto, config change); the driver's PEC restarts there.

In [ ]:
# ── PEC vs Guider: running totals, rates and PEC rate terms per axis, from PECLOG ─────────────────────
pvg = correction_breakdown(df, rate_half_window_s=300)
C = {'mount drift': 'white', 'pulse-guide': 'royalblue', 'sync-guide': 'orange', 'PEC applied': 'magenta'}
summary = []
for ax, name in (('ra', 'RA'), ('dec', 'Dec')):
    x = pvg[ax]
    if x.empty:
        print(f"{name}: no guide corrections in PECLOG")
        continue
    tx = x['timestamp']
    fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.06,
        subplot_titles=[f'{name} 1. running totals (arcsec): mount drift = pulse-guide + sync-guide + PEC applied',
                        f'{name} 2. rates (arcsec/min, ±5 min slopes): drift rate = pulse + sync + PEC rates',
                        f'{name} 3. PEC rate terms (arcsec/min): model rate = steady-drift term + harmonic contribution'])
    trend = np.polyval(np.polyfit(x['t_sec'], x['mount_drift'], 2), x['t_sec'])
    for col, label in (('mount_drift', 'mount drift'), ('pulse_guide', 'pulse-guide'), ('sync_guide', 'sync-guide'), ('pec_applied', 'PEC applied')):
        nm = label if col == 'mount_drift' else (f'{label} corrections' if label != 'PEC applied' else label)
        fig.add_trace(go.Scatter(x=tx, y=x[col], name=f'{name} {nm}', legendgroup=label,
                                 line=dict(color=C[label], width=3 if col == 'mount_drift' else 1.5)), row=1, col=1)
        rate = 'drift_rate' if col == 'mount_drift' else f'{col}_rate'
        rnm = 'actual drift rate (±5 min)' if col == 'mount_drift' else (f'{label} rate' if label != 'PEC applied' else 'PEC applied rate (±5 min)')
        fig.add_trace(go.Scatter(x=tx, y=x[rate], name=f'{name} {rnm}', legendgroup=label, showlegend=True,
                                 line=dict(color=C[label], width=3 if col == 'mount_drift' else 1.5)), row=2, col=1)
    fig.add_trace(go.Scatter(x=tx, y=trend, name=f'{name} mount drift quadratic trend', line=dict(color='gray', dash='dash')), row=1, col=1)
    fig.add_trace(go.Scatter(x=tx, y=x['drift_rate'], name=f'{name} actual drift rate (±5 min)', showlegend=False,
                             line=dict(color='white', width=3)), row=3, col=1)
    fig.add_trace(go.Scatter(x=tx, y=x['pec_model_rate'], name=f'{name} PEC model rate', line=dict(color='red', width=2)), row=3, col=1)
    fig.add_trace(go.Scatter(x=tx, y=x['pec_steady_term'], name=f'{name} PEC steady-drift term', line=dict(color='#636EFA', dash='dash')), row=3, col=1)
    if np.nanmax(np.abs(x['pec_harmonic_term'])) > 1e-6:
        fig.add_trace(go.Scatter(x=tx, y=x['pec_harmonic_term'], name=f'{name} PEC harmonic contribution (signed)',
                                 line=dict(color='violet', dash='dash')), row=3, col=1)
    fig.add_trace(go.Scatter(x=tx, y=np.where(x['pec_active'], x['pec_applied_logged_rate'], 0.0), name=f'{name} PEC applied rate',
                             line=dict(color='magenta', width=1.5)), row=3, col=1)
    # shade where PEC was inhibited (not applying)
    inactive = ~x['pec_active'].values.astype(bool)
    edges = np.flatnonzero(np.diff(np.r_[0, inactive.astype(int), 0]))
    for a, b in zip(edges[::2], edges[1::2]):
        fig.add_vrect(x0=tx.iloc[a], x1=tx.iloc[min(b, len(tx) - 1)], fillcolor='gray', opacity=0.15, line_width=0, row=3, col=1)
    for r in (2, 3):
        fig.add_hline(y=0, line=dict(color='gray', width=1), row=r, col=1)
    for tr in pvg['resets']:
        fig.add_vline(x=tr, line=dict(color='gray', width=1, dash='dot'))
    fig.update_layout(height=1000, width=1400, template='plotly_dark', hovermode='x unified',
                      title=f'{name}: who corrected the drift (shaded in 3: PEC inhibited; dotted: PEC model resets)')
    fig.show()

    last = x.iloc[-1]
    guider = last['pulse_guide'] + last['sync_guide']
    pec = last['pec_applied']
    m = ~np.isnan(x['drift_rate'].values)
    err = (x['pec_model_rate'] - x['drift_rate'])[m]
    summary.append({'axis': name, 'mount drift "': round(last['mount_drift']), 'pulse-guide "': round(last['pulse_guide']),
                    'sync-guide "': round(last['sync_guide']), 'PEC applied "': round(pec),
                    'PEC share of drift %': round(pec / last['mount_drift'] * 100) if last['mount_drift'] else np.nan,
                    'PEC undone by guider %': round(-guider / pec * 100) if pec and np.sign(guider) != np.sign(pec) else 0,
                    'PEC active %': round(x['pec_active'].mean() * 100),
                    'model rate error rms "/min': round(float(np.sqrt(np.mean(err ** 2))), 1),
                    'actual drift rate rms "/min': round(float(np.sqrt(np.nanmean(x['drift_rate'] ** 2))), 1)})
summary = pd.DataFrame(summary)
display(summary)
for _, r in summary.iterrows():
    if r['PEC share of drift %'] < 0:
        print(f"{r['axis']}: PEC applied {r['PEC applied \"']:+}\" against a mount drift of {r['mount drift \"']:+}\" -- it pushed the opposite way to the drift")
    if r['PEC undone by guider %'] > 50:
        print(f"{r['axis']}: the guider undid {r['PEC undone by guider %']}% of what PEC applied -- PEC and the guider are fighting")
    if r['model rate error rms "/min'] > r['actual drift rate rms "/min']:
        print(f"{r['axis']}: PEC's model rate was further from the actual drift rate than zero would have been")

## Guider Outcome (optional -- needs the PHD2 guide log)

Purpose: did guiding get better, and did the guider have to work less, once PEC was on? Uses only the
PHD2 guide log, in time blocks (`BLOCK`), so PEC-off and PEC-on periods on the same target can be compared.

- **Top chart -- guiding RMS (arcsec):** RA and Dec bars from PHD2's raw guide-star error, white line the
  combined total. Lower is better.
- **Bottom chart -- guider effort (pulse ms per minute):** total RA and Dec pulse time in both directions
  (at 1x guide rate, 1000 ms = 15"). If PEC is carrying the mount's drift, these bars should shrink once
  PEC is on. Effort includes chasing seeing, so the net direction is in the PEC vs Guider chart instead.
- **Shading** marks blocks with PEC on; **orange dotted lines** are plate-solve syncs. Each sync moves the
  model and the guider pulls the star back, so the summary also reports RMS excluding the minutes after
  each sync (`SYNC_SETTLE`).

Set `PHD2_GUIDELOG = None` to skip.


In [ ]:
# ── Guider outcome from the PHD2 guide log (optional) ─────────────────────────────────────────
BLOCK       = '20min'                  # time block for the table/chart
SYNC_SETTLE = pd.Timedelta('3min')     # frames this long after a sync are excluded from the on/off comparison

if not PHD2_GUIDELOG:
    print("No PHD2 guide log (PHD2_GUIDELOG = None) -- guider outcome skipped.")
else:
    phd2_path = PHD2_GUIDELOG if os.path.isabs(PHD2_GUIDELOG) else os.path.join(LOG_DIR, PHD2_GUIDELOG)
    phd2_frames, phd2_events = load_phd2_guidelog(phd2_path)
    phd2_frames = window_df(phd2_frames, T_FROM, T_TO)
    phd2_events = window_df(phd2_events, T_FROM, T_TO)
    if phd2_frames.empty:
        print(f"No PHD2 guided frames inside the window T_FROM={T_FROM!r} T_TO={T_TO!r}")
    else:
        fr = phd2_frames.copy()
        active = df[df['pec_active'].astype(bool)] if 'pec_active' in df.columns else df
        on = pd.merge_asof(fr[['timestamp']].astype('datetime64[ns]'),
                           active[['timestamp']].astype('datetime64[ns]').assign(pec_on=True),
                           on='timestamp', direction='nearest', tolerance=pd.Timedelta('60s'))
        fr['pec_on'] = on['pec_on'].astype('boolean').fillna(False).astype(bool).values
        # syncs from the always-logged SYNC GUIDING lines, so syncs while PEC was off count too
        try:
            sync_times = window_df(load_sync_guiding_residuals(log_filenames, log_dir=LOG_DIR), T_FROM, T_TO)['timestamp']
        except ValueError:
            sync_times = pd.Series([], dtype='datetime64[ns]')
        fr['after_sync'] = False
        for t in sync_times:
            fr.loc[(fr['timestamp'] >= t) & (fr['timestamp'] < t + SYNC_SETTLE), 'after_sync'] = True

        rms = lambda s: float(np.sqrt(np.mean(np.square(s)))) if len(s) else np.nan
        fr['block'] = fr['timestamp'].dt.floor(BLOCK)
        g = fr.groupby('block')
        minutes = g['timestamp'].agg(lambda s: max((s.iloc[-1] - s.iloc[0]).total_seconds() / 60, 1e-9))
        lost = phd2_events[phd2_events['kind'] == 'star_lost'].groupby(
            phd2_events['timestamp'].dt.floor(BLOCK)).size() if not phd2_events.empty else pd.Series(dtype=int)
        blocks = pd.DataFrame({
            'PEC on %':        (g['pec_on'].mean() * 100).round(0),
            'RA rms "':        g['ra_arcsec'].agg(rms).round(2),
            'Dec rms "':       g['dec_arcsec'].agg(rms).round(2),
            'total rms "':     g.apply(lambda x: rms(np.hypot(x['ra_arcsec'], x['dec_arcsec'])), include_groups=False).round(2),
            'RA pulses ms/min':  (g['ra_ms'].agg(lambda s: s.abs().sum()) / minutes).round(0),
            'Dec pulses ms/min': (g['dec_ms'].agg(lambda s: s.abs().sum()) / minutes).round(0),
            'syncs':           sync_times.groupby(sync_times.dt.floor(BLOCK)).size(),
            'star lost':       lost,
        }).loc[minutes.index].fillna({'syncs': 0, 'star lost': 0}).astype({'syncs': int, 'star lost': int})
        print(blocks.to_string())
        print()

        for label, sel in (('PEC off', ~fr['pec_on']), ('PEC on', fr['pec_on'])):
            for excl, mask in (('all frames', sel), (f'excluding {SYNC_SETTLE.total_seconds()/60:.0f} min after syncs', sel & ~fr['after_sync'])):
                x = fr[mask]
                if len(x):
                    print(f"{label:7s} {excl:30s} RA {rms(x['ra_arcsec']):.2f}\"  Dec {rms(x['dec_arcsec']):.2f}\"  "
                          f"total {rms(np.hypot(x['ra_arcsec'], x['dec_arcsec'])):.2f}\"  ({len(x)} frames)")

        fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
            subplot_titles=[f'Guiding RMS per {BLOCK} (arcsec)', 'Guider effort (pulse ms per minute)'])
        fig.add_trace(go.Bar(x=blocks.index, y=blocks['RA rms "'], name='RA rms', marker_color='royalblue'), row=1, col=1)
        fig.add_trace(go.Bar(x=blocks.index, y=blocks['Dec rms "'], name='Dec rms', marker_color='mediumseagreen'), row=1, col=1)
        fig.add_trace(go.Scatter(x=blocks.index, y=blocks['total rms "'], name='total rms', mode='lines+markers',
                                 line=dict(color='white')), row=1, col=1)
        fig.add_trace(go.Bar(x=blocks.index, y=blocks['RA pulses ms/min'], name='RA pulses', marker_color='royalblue',
                             showlegend=False), row=2, col=1)
        fig.add_trace(go.Bar(x=blocks.index, y=blocks['Dec pulses ms/min'], name='Dec pulses', marker_color='mediumseagreen',
                             showlegend=False), row=2, col=1)
        for t in sync_times:
            fig.add_vline(x=t, line=dict(color='orange', width=1, dash='dot'))
        on_blocks = blocks.index[blocks['PEC on %'] >= 50]
        for b in on_blocks:
            fig.add_vrect(x0=b, x1=b + pd.Timedelta(BLOCK), fillcolor='magenta', opacity=0.08, line_width=0)
        fig.update_layout(height=750, width=1400, template='plotly_dark', barmode='group', hovermode='x unified',
            title='Guider Outcome (shaded: PEC on, dotted: syncs)')
        fig.show()


## PEC rate benchmark (this session)

The driver's real `PecAxis` replayed causally on this session's drift (`total_accum`, joined across PEC model resets):
what each model *would have* applied at each moment, scored by the **drift left for the guider per correction
interval** (30 s ~ PHD2, 2-5 min ~ plate-solve sync guiding), relative to no PEC. Lower is better; below 1 means PEC
would have helped. "This session's PEC" uses the PECCONFIG logged for the session. Same method as
`analyse_pec_catalog.ipynb`, which runs it over every archived session.

In [ ]:
# ── PEC rate benchmark for this session ─────────────────────────────────────────────────────────
BENCH_INTERVALS = (30, 120, 300)        # s
THIN_S = 5.0                            # pulse-guided PECLOG is ~1 Hz; PecAxis uses each sample's own dt
d = df.dropna(subset=['total_accum_1', 'total_accum_2']).sort_values('t_sec')
reset = np.r_[False, np.diff(d['n'].values) < 0]                        # PEC model resets: accumulator restarts
t_all = d['t_sec'].values
drift = {'ra': stitch_accum(d['total_accum_1'].values * 60, reset),     # arcmin -> arcsec
         'dec': stitch_accum(d['total_accum_2'].values * 60, reset)}
keep = np.r_[True, np.diff(np.floor(t_all / THIN_S)) > 0]
t_b = t_all[keep] - t_all[0]

mode = pec_config['mode'] if pec_config['mode'] in ('rls', 'ema') else 'rls'
session_model = PecModel(f"this session's PEC ({mode}, H={pec_config['H'] if mode == 'rls' else 0}, tau {pec_config['tau'] / 60:.0f}m)",
                         mode, int(pec_config['H']) if mode == 'rls' else 0, float(pec_config['tau']), float(pec_config['T']))
BENCH_MODELS = [session_model, PecModel('RLS linear, tau 35m', 'rls', 0, 2100)] + \
               [PecModel(f'EMA tau {m}m', 'ema', 0, m * 60) for m in (1, 2, 5, 10, 15, 21, 30, 60)]
bench_rates, rows = {}, []
for ax in ('ra', 'dec'):
    y = drift[ax][keep]
    rates = {m.name: replay_pec_rate(t_b, y, m) for m in BENCH_MODELS}
    bench_rates[ax] = (y, rates)
    sc = rate_scores(t_b, y, rates, guide_intervals_s=BENCH_INTERVALS, ref_half_window_s=300)
    for dt, res in sc['guide_rms'].items():
        rows += [dict(axis=ax, interval=f'{dt} s', model=name, rms=v) for name, v in res.items()]
bench = pd.DataFrame(rows)
pooled = bench.pivot_table(index='model', columns='interval', values='rms', aggfunc=lambda v: np.sqrt(np.mean(np.square(v))))
rel = pooled.div(pooled.loc['no_pec'], axis=1).reindex(['no_pec'] + [m.name for m in BENCH_MODELS])[[f'{dt} s' for dt in BENCH_INTERVALS]]
print(f"{t_b[-1] / 3600:.1f} h of drift, {len(t_b)} samples.  Drift left for the guider per interval, relative to no PEC:")
display(rel.round(3))
ema_rows = [m.name for m in BENCH_MODELS if m.mode == 'ema']
best = rel.loc[ema_rows, '120 s'].idxmin()
print(f"best EMA for a 2-min interval: {best} ({rel.loc[best, '120 s']:.2f}); this session's PEC: {rel.loc[session_model.name, '120 s']:.2f}")

In [ ]:
# ── Actual drift rate vs what each model would have applied ─────────────────────────────────────
AXIS = 'ra'
SHOW = [session_model.name, 'EMA tau 5m', 'EMA tau 21m']
y, rates = bench_rates[AXIS]
fig = go.Figure()
fig.add_trace(go.Scatter(x=t_b / 60, y=true_rate(t_b, y, 300), name=f'{AXIS.upper()} actual drift rate (±5 min)', line=dict(color='white', width=3)))
for name, c in zip(SHOW, ('red', 'mediumseagreen', 'royalblue')):
    fig.add_trace(go.Scatter(x=t_b / 60, y=rates[name], name=name, line=dict(color=c)))
fig.update_layout(height=450, width=1400, template='plotly_dark', hovermode='x unified',
                  title=f'{AXIS.upper()} drift rate: actual vs applied by each model', xaxis_title='minutes', yaxis_title='arcsec/min')
fig.show()

## Base vs Corrected Frame Shift (optional -- requires KFLOG)

`θ_meas_raw` is the raw motor angle (from 518, before any correction). `θ_ref_raw` is the PID's target in
**corrected** motor space: the PID drives the corrected angle to it, and MAC, pulse-guide, sync-guide and PEC
corrections are deliberately *not* undone. So

**shift = θ_meas_raw − θ_ref_raw ≈ −(every correction applied so far, as motor angles) + the PID's tracking error.**

With tracking only and no guiding the shift stays within a few arcsec. Once corrections are applied it moves, and
how it moves tells you who corrected what. The kinds of divergence it shows:

1. **Staircase at each sync.** A plate-solve sync applies its residual as an instant step in the corrected angle;
   the PID moves the motors to follow, so the shift steps by the residual projected onto each motor.
2. **Ramps between syncs.** Anything applied continuously: PEC's correction (fed through the sync-guide rotation
   every control tick), MAC changing with pose, slow PID error. With PEC off this is close to flat, and all the
   correction arrives as steps.
3. **Ramp undone by the next sync -- PEC fighting.** If PEC's rate were right, its ramp would go *with* the drift
   and the next sync step would be small. When the next step reverses the ramp (correlation near −1, *share undone*
   near 1), each plate solve is taking back what PEC applied: PEC is pushing the wrong way. In the archive this shows
   up only with PEC on, and was the first sign PEC wasn't working as intended.
4. **Mirror-image M1 and M3.** When M2 (θ2) is small the M1 and M3 axes nearly line up, and a small correction on
   the sky needs large, opposite M1/M3 rotations (about 1 / sin θ2). Big counter-rotating M1/M3 shifts there are
   geometry, not mount error.
5. **Fast ripple and spikes (seconds).** PID tracking error, the speed controller's dither, M3's 0.6 s
   sample-and-hold in 518, and settling transients right after each sync step.

Gotos and slews while tracking appear as jumps of degrees; they're removed from the series and the minute after
each is left out of the ripple. For a session with several targets, set `T_FROM`/`T_TO` to one tracking run for
clean totals (the ramp-vs-next-sync test is unaffected).

KFLOG is gated by `Config.log_position` (independent of `log_pec`); this section is skipped when it's missing.
See the commented KFLOG sessions in the log-selection cell.

In [ ]:
# ── Base vs Corrected Frame Shift per motor, from KFLOG ─────────────────────────────────────────
kf_df, _pid_df_unused = load_kf_pid(log_filenames, log_dir=LOG_DIR)
kf_df = window_df(kf_df, T_FROM, T_TO)
_shift_cols = [f'\u03b8_meas_raw_{i}' for i in (1, 2, 3)] + [f'\u03b8_ref_raw_{i}' for i in (1, 2, 3)]
if not len(kf_df) or not all(c in kf_df.columns for c in _shift_cols):
    print("No KFLOG θ_meas_raw/θ_ref_raw in this capture -- frame shift diagnostic skipped "
          "(needs Config.log_position=true; see the commented KFLOG sessions in the log-selection cell).")
else:
    kf = kf_df.dropna(subset=_shift_cols).sort_values('timestamp').reset_index(drop=True)
    try:
        kf_syncs = window_df(load_sync_guiding_residuals(log_filenames, log_dir=LOG_DIR), T_FROM, T_TO)['timestamp']
    except ValueError:
        kf_syncs = pd.Series([], dtype='datetime64[ns]')
    kf_syncs = kf_syncs[(kf_syncs >= kf['timestamp'].iloc[0]) & (kf_syncs <= kf['timestamp'].iloc[-1])]
    comp, shift_summary = frame_shift_components(kf, kf_syncs)
    amp = shift_summary.attrs['m1_m3_amplification']
    print(f"{len(kf)} KFLOG samples over {comp['t_min'].iloc[-1]:.0f} min, {len(kf_syncs)} syncs. "
          f"{shift_summary.attrs['breaks']} goto/slew jumps removed. "
          f"Median θ2 {shift_summary.attrs['theta2_deg']:.1f}° -> M1/M3 amplification x{amp:.1f}"
          + ("  (mirror-image M1/M3 shifts expected)" if amp > 2 else ""))
    display(shift_summary.round(1))

    fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.06,
                        subplot_titles=[f'M{i}: shift (arcsec) = staircase at syncs + change between syncs' for i in (1, 2, 3)])
    for i in (1, 2, 3):
        m = f'M{i}'
        fig.add_trace(go.Scatter(x=comp['timestamp'], y=comp[f'{m} shift'], name=f'{m} shift', line=dict(color='gray', width=1),
                                 legendgroup=m), row=i, col=1)
        fig.add_trace(go.Scatter(x=comp['timestamp'], y=comp[f'{m} staircase'] + comp[f'{m} shift'].iloc[:50].mean(),
                                 name=f'{m} staircase (syncs)', line=dict(color='orange', width=2), legendgroup=m), row=i, col=1)
        fig.add_trace(go.Scatter(x=comp['timestamp'], y=comp[f'{m} smooth'], name=f'{m} between syncs', line=dict(color='cyan', width=2),
                                 legendgroup=m), row=i, col=1)
    fig.update_layout(height=900, width=1400, template='plotly_dark', hovermode='x unified',
                      title='Base vs Corrected Frame Shift: steps at syncs (orange) vs change between syncs (cyan)')
    fig.show()

    fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.06,
                        subplot_titles=[f'M{i}: fast ripple (shift minus its 60 s rolling median, arcsec)' for i in (1, 2, 3)])
    for i in (1, 2, 3):
        fig.add_trace(go.Scatter(x=comp['timestamp'], y=comp[f'M{i} ripple'], name=f'M{i} ripple', line=dict(width=1)), row=i, col=1)
    fig.update_layout(height=650, width=1400, template='plotly_dark', hovermode='x unified')
    fig.show()

    rvs = ramp_vs_next_step(comp, kf_syncs)
    print("Change between syncs (ramp) vs the step at the next sync, per motor:")
    display(rvs.round(2))
    for m, r in rvs.iterrows():
        if r['intervals'] >= 5 and r['corr'] < -0.5 and r['share undone'] > 0.5:
            print(f"{m}: the next sync undoes {r['share undone'] * 100:.0f}% of each ramp (corr {r['corr']:+.2f}) -- "
                  f"what's applied between syncs (PEC) is pushing against the drift")

# PEC model internals (replay)

The session's own PEC model (from the logged PECCONFIG) replayed on the mount drift, for the model's fit-quality
diagnostics -- R² and fit rmse, which drive PEC's inhibit (PEC stops applying when R² < 0.5). These describe how well
the model fits the *past* drift, not whether its current rate is right; for that, see PEC vs Guider (3) and the PEC
rate benchmark.

In [ ]:
# ── replay the session's PEC model on the mount drift (for the fit-quality charts below) ─────
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from control import PecAxis, PecInhibit, PecMode

# ── calc INTERVAL (diagnostic only — no longer drives lambda/alpha directly) ──
ra_times = df.dropna(subset=['resid_1'])['t_sec']
INTERVAL = ra_times.diff().dropna().median()

# ── instantiate from parsed session config ────────────────────────────────
T        = pec_config['T']
H        = pec_config['H']
TAU      = pec_config['tau']
MIN_DT   = pec_config['min_dt']
MODE     = PecMode(pec_config['mode']) if pec_config['mode'] in ('rls', 'ema') else PecMode.RLS
VAR      = 0.05
SSE      = 0.15
COLORS   = ['green', 'purple', 'orange', 'red', 'cyan', 'magenta']
DEG_S_TO_ARCMIN_HR = 3600 * 60   # deg/sec to arcmin/hr

ra  = PecAxis(T=T, n_harmonics=H, mode=MODE, tau=TAU, min_dt=MIN_DT)
dec = PecAxis(T=T, n_harmonics=H, mode=MODE, tau=TAU, min_dt=MIN_DT)
var_alpha = VAR
sse_alpha = SSE

ra.reset_seed(df.iloc[0].total_accum_1 / 60)
dec.reset_seed(df.iloc[0].total_accum_2 / 60)

# mount drift joined across PEC model resets (total_accum restarts at 0 at each reset), arcsec -- as in PEC vs Guider
_reset = np.r_[False, np.diff(df['n'].values) < 0]
ra_drift_arcsec  = stitch_accum(df['total_accum_1'].values * 60, _reset)
dec_drift_arcsec = stitch_accum(df['total_accum_2'].values * 60, _reset)
ra_offset  = ra_drift_arcsec  - df['total_accum_1'].values * 60     # per-row offset added by the join
dec_offset = dec_drift_arcsec - df['total_accum_2'].values * 60
ARCSEC_PER_MIN = 3600 * 60          # deg/sec -> arcsec/min (numerically = arcmin/hr)

t0 = 0.0            # local time origin — shifts forward on every detected PEC reset
prev_n = df.iloc[0].n

show_harmonics = (MODE == PecMode.RLS and H > 0)

records = []
reset_events = []   # for plotting/inspection — t_min of each detected reset
reset_indices = []  # for fixing gradient

for i, (_, row) in enumerate(df.iterrows()):
    t_abs     = row.t_sec
    ra_cumul  = row.total_accum_1  / 60
    dec_cumul = row.total_accum_2 / 60

    # ── detect a PEC reset from the guide counter rolling back ──
    if row.n < prev_n:
        ra.reset()
        dec.reset()
        ra.reset_seed(ra_cumul)
        dec.reset_seed(dec_cumul)
        t0 = t_abs                      # restart local clock — matches driver's _pec_t0 reset
        reset_events.append(t_abs / 60)
        reset_indices.append(i)
    prev_n = row.n

    t = t_abs - t0                      # local time, relative to most recent reset

    ra.ingest_accum(ra_cumul, t, var_alpha, sse_alpha)
    dec.ingest_accum(dec_cumul, t, var_alpha, sse_alpha)

    records.append(dict(
        t_min    = t_abs / 60,          # keep x-axis on session-absolute time for plotting continuity
        timestamp = row.timestamp,
        n        = row.n,
        ra_rate  = ra.predicted_rate(t)  * ARCSEC_PER_MIN,
        dec_rate = dec.predicted_rate(t) * ARCSEC_PER_MIN,
        ra_drift = ra.dc_rate()   * ARCSEC_PER_MIN,
        dec_drift= dec.dc_rate()  * ARCSEC_PER_MIN,
        **{f'ra_h{h}':  ra.harmonic_rate(h)  * ARCSEC_PER_MIN for h in range(1, H+1)},
        **{f'dec_h{h}': dec.harmonic_rate(h) * ARCSEC_PER_MIN for h in range(1, H+1)},
        ra_r2    = ra.r2,
        dec_r2   = dec.r2,
        ra_rmse  = ra.rmse_arcmin() * 60,
        dec_rmse = dec.rmse_arcmin() * 60,
        ra_pred  = ra.predicted_accum(t)  * 3600 + ra_offset[i],      # arcsec, on the joined mount drift
        dec_pred = dec.predicted_accum(t) * 3600 + dec_offset[i],
        ra_cumul = ra_drift_arcsec[i],
        dec_cumul= dec_drift_arcsec[i],
        ra_resid_fit  = (ra_cumul  - ra.predicted_accum(t))  * 3600,
        dec_resid_fit = (dec_cumul - dec.predicted_accum(t)) * 3600,
        ra_lambda  = ra.lam,
        dec_lambda = dec.lam,
    ))

res = pd.DataFrame(records)
t   = res.t_min
t   = res.timestamp

# ── empirical (actual) drift rate ──
res['ra_actual_rate']  = np.full(len(res), np.nan)
res['dec_actual_rate'] = np.full(len(res), np.nan)

# slope of the joined mount drift over +-5 min (arcsec/min), the same "actual drift rate" as the benchmark
res['ra_actual_rate']  = true_rate(res.t_min.values * 60, res.ra_cumul.values,  300)
res['dec_actual_rate'] = true_rate(res.t_min.values * 60, res.dec_cumul.values, 300)

mode_label = MODE.value.upper()
title_text = (f'PEC Model (mode={mode_label}   Hₙ={H}   Period₁={T/60:.0f} min   '
              f'dt_med={INTERVAL:.1f}s   τ={TAU:.0f}s   sse_α={SSE}   var_α={VAR})')
print(title_text)


In [ ]:
# ── PEC model fit quality: R² and mount drift minus fit ─────────────────────────────────────
row5_min = 0
n_rows_bottom = 2
row_titles_bottom = [
    'RA fit R² quality',                      'Dec fit R² quality',
    'RA mount drift minus PEC model fit, and fit rmse (arcsec)', 'Dec mount drift minus PEC model fit, and fit rmse (arcsec)',
]

fig_bottom = make_subplots(
    rows=n_rows_bottom, cols=2,
    shared_xaxes=True,
    subplot_titles=tuple(row_titles_bottom),
    vertical_spacing=0.12,
)

def add(fig, row, col, traces):
    for tr in traces:
        fig.add_trace(tr, row=row, col=col)

# row 1 (of this fig) — R²
add(fig_bottom, 1, 1, [go.Scatter(x=t, y=res.ra_r2,  name='RA R²',  line=dict(color='royalblue'))])
add(fig_bottom, 1, 2, [go.Scatter(x=t, y=res.dec_r2, name='Dec R²', line=dict(color='royalblue'))])

# row 2 (of this fig) — residuals
add(fig_bottom, 2, 1, [
    go.Scatter(x=t, y=res.ra_resid_fit, name='RA mount drift minus fit', line=dict(color='royalblue')),
    go.Scatter(x=t, y=res.ra_rmse,      name='RA fit rmse',      line=dict(color='red', dash='dash')),
])
add(fig_bottom, 2, 2, [
    go.Scatter(x=t, y=res.dec_resid_fit, name='Dec mount drift minus fit', line=dict(color='royalblue')),
    go.Scatter(x=t, y=res.dec_rmse,      name='Dec fit rmse',      line=dict(color='red', dash='dash')),
])

fig_bottom.add_hrect(y0=row5_min, y1=0.5, fillcolor='rgba(255,0,0,0.15)', line_width=0, row=1, col=1)
fig_bottom.add_hrect(y0=row5_min, y1=0.5, fillcolor='rgba(255,0,0,0.15)', line_width=0, row=1, col=2)
fig_bottom.update_yaxes(range=[row5_min, 1.0], row=1, col=1)
fig_bottom.update_yaxes(range=[row5_min, 1.0], row=1, col=2)
fig_bottom.update_xaxes(matches="x")   # link every subplot's x-axis to the first, across both columns

fig_bottom.update_layout(
    height=300 * n_rows_bottom,
    title=dict(text=title_text, x=0.5),
    hovermode='x unified', template="plotly_dark",
    legend=dict(groupclick='toggleitem'),
)
fig_bottom.update_xaxes(title_text='Time (minutes)', row=n_rows_bottom)
fig_bottom.show()


# Right Ascension Drift

In [ ]:
x = pvg['ra']
t = x['t_sec'].values; y = x['mount_drift'].values              # arcsec, joined across PEC resets
(slope, intercept), residuals, *_ = np.polyfit(t, y, 1, full=True)
fit = np.polyval([slope, intercept], t)
rss = float(np.sum((y - fit) ** 2)); tss = float(np.sum((y - y.mean()) ** 2))
r2 = 1 - rss / tss if tss else float('nan')
print(f"=== Right Ascension - mount drift linear fit (N={len(t)}) ===")
print(f"Slope (drift rate): {slope * 60:+8.2f} arcsec/min")
print(f"RMSE (noise):       {np.sqrt(rss / len(t)):8.1f} arcsec")
print(f"R² (fit quality):   {r2:8.3f} {r2_quality(r2)} fit")

# Declination Drift

In [ ]:
x = pvg['dec']
t = x['t_sec'].values; y = x['mount_drift'].values              # arcsec, joined across PEC resets
(slope, intercept), residuals, *_ = np.polyfit(t, y, 1, full=True)
fit = np.polyval([slope, intercept], t)
rss = float(np.sum((y - fit) ** 2)); tss = float(np.sum((y - y.mean()) ** 2))
r2 = 1 - rss / tss if tss else float('nan')
print(f"=== Declination - mount drift linear fit (N={len(t)}) ===")
print(f"Slope (drift rate): {slope * 60:+8.2f} arcsec/min")
print(f"RMSE (noise):       {np.sqrt(rss / len(t)):8.1f} arcsec")
print(f"R² (fit quality):   {r2:8.3f} {r2_quality(r2)} fit")

# Right Ascension Period

In [ ]:
x = pvg['ra']
t = x['t_sec'].values; y = x['mount_drift'].values
periods, power = lombscargle_periodogram(t, y, min_period=5 * 60, max_period=min(120 * 60, (t[-1] - t[0]) / 2))
k = int(np.argmax(power)); peak = periods[k]
noise = float(np.median(power)); snr_db = 10 * np.log10(power[k] / noise) if noise > 0 else np.inf
detr = y - np.polyval(np.polyfit(t, y, 2), t)
print(f"=== RA - strongest period in the mount drift (N={len(t)}, Lomb-Scargle, quadratic trend removed) ===")
print(f"Period:             {peak / 60:8.1f} min")
print(f"Amplitude (half p-p of detrended drift): {np.ptp(detr) / 2:6.1f} arcsec")
print(f"SNR (peak vs median power): {snr_db:6.1f} dB {pec_quality_db(snr_db)} signal")

In [ ]:
fig = go.Figure(go.Scatter(x=periods / 60, y=power, mode='lines', name='RA power'))
fig.add_vline(x=peak / 60, line_dash='dash', line_color='red')
fig.update_layout(title='RA mount drift periodogram (Lomb-Scargle)', xaxis_title='Period (minutes)', yaxis_title='normalised power',
                  height=500, width=1200, template='plotly_dark')
fig.show()

# Declination Period

In [ ]:
x = pvg['dec']
t = x['t_sec'].values; y = x['mount_drift'].values
periods, power = lombscargle_periodogram(t, y, min_period=5 * 60, max_period=min(120 * 60, (t[-1] - t[0]) / 2))
k = int(np.argmax(power)); peak = periods[k]
noise = float(np.median(power)); snr_db = 10 * np.log10(power[k] / noise) if noise > 0 else np.inf
detr = y - np.polyval(np.polyfit(t, y, 2), t)
print(f"=== Dec - strongest period in the mount drift (N={len(t)}, Lomb-Scargle, quadratic trend removed) ===")
print(f"Period:             {peak / 60:8.1f} min")
print(f"Amplitude (half p-p of detrended drift): {np.ptp(detr) / 2:6.1f} arcsec")
print(f"SNR (peak vs median power): {snr_db:6.1f} dB {pec_quality_db(snr_db)} signal")

In [ ]:
fig = go.Figure(go.Scatter(x=periods / 60, y=power, mode='lines', name='Dec power'))
fig.add_vline(x=peak / 60, line_dash='dash', line_color='red')
fig.update_layout(title='Dec mount drift periodogram (Lomb-Scargle)', xaxis_title='Period (minutes)', yaxis_title='normalised power',
                  height=500, width=1200, template='plotly_dark')
fig.show()

# Notes